# 검색 베이스라인 확인

Colab의 CPU 런타임에서 위에서부터 실행합니다. GPU는 필요하지 않습니다.
공개 가상 문서 3건을 사용하며 팀 원본 데이터는 포함하지 않습니다.
프로젝트 의존성은 uv.lock으로 설치합니다. 코드 파일을 노트북에 복사하지 않습니다.
마지막 셀에서 질문을 직접 입력할 수 있습니다.


In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

# uv 도구만 설치하고, 프로젝트 패키지는 아래 uv sync로 맞춥니다.
uv = shutil.which("uv")
if uv is None:
    subprocess.run(["bash", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
    uv = str(Path.home() / ".local/bin/uv")

project = Path("/content/rfp-retrieval-baseline")
branch = "retrieval/day2-sample-baseline"
repository = "https://github.com/yulesn/rfp-agent-part3-team1.git"
if not project.exists():
    subprocess.run(["git", "clone", "--branch", branch, "--single-branch", repository, str(project)], check=True)
else:
    current = subprocess.check_output(["git", "branch", "--show-current"], cwd=project, text=True).strip()
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=project, text=True).strip()
    if current != branch or remote != repository:
        raise RuntimeError("다른 저장소 또는 브랜치가 있습니다. 새 런타임에서 시작해 주세요.")
    subprocess.run(["git", "pull", "--ff-only"], cwd=project, check=True)

subprocess.run([uv, "sync", "--frozen", "--python", "3.11"], cwd=project, check=True)

def run_project(*arguments, capture=False):
    return subprocess.run(
        [uv, "run", "--frozen", "--python", "3.11", *arguments],
        cwd=project, check=True, text=True, capture_output=capture,
    )


## 1. API 없이 확인

자동 테스트와 청크 입력 검사는 API 키 없이 실행할 수 있습니다.
자동 테스트용 벡터는 실제 의미 검색 품질을 검증하지 않습니다.


In [ ]:
run_project("pytest", "-q")
run_project("python", "scripts/retrieval_index.py", "--config", "configs/retrieval_demo.yaml", "validate", "--chunks", "examples/retrieval_demo/chunks.jsonl")


## 2. 실제 임베딩 연결

Colab 왼쪽 열쇠 메뉴에 OPENAI_API_KEY를 등록하고 이 노트북의 접근을 허용합니다.
등록하지 않았다면 숨김 입력창에 키를 입력할 수 있습니다. 코드나 출력에 키를 남기지 않습니다.
다음 DB 생성과 검색은 실제 API를 호출합니다.


In [ ]:
from getpass import getpass
from google.colab import userdata

try:
    team_key = userdata.get("OPENAI_API_KEY")
except Exception:
    team_key = getpass("팀 OpenAI API 키: ")
if not team_key or not team_key.strip():
    raise ValueError("API 키가 필요합니다.")
os.environ["OPENAI_API_KEY"] = team_key.strip()
del team_key
print("키 설정 완료. 키 값은 출력하지 않습니다.")


In [ ]:
# 이미 완성된 예제 DB가 있으면 다시 임베딩하지 않습니다.
status = run_project("python", "scripts/retrieval_index.py", "--config", "configs/retrieval_demo.yaml", "inspect", capture=True)
state = json.loads(status.stdout)["state"]
if state == "missing":
    run_project("python", "scripts/retrieval_index.py", "--config", "configs/retrieval_demo.yaml", "build", "--chunks", "examples/retrieval_demo/chunks.jsonl", "--report", "data/processed/retrieval_demo/build-report.json")
elif state != "ready":
    raise RuntimeError("완성되지 않은 DB입니다. 실행 가이드의 재생성 절차를 확인해 주세요.")
run_project("python", "scripts/retrieval_index.py", "--config", "configs/retrieval_demo.yaml", "inspect")


## 3. 기대한 근거가 검색되는지 확인

예제 질문 3건의 1순위 문서와 근거 문장을 검사합니다.
이 결과는 작은 예제의 연결 확인이며 팀 전체 문서의 검색 성능을 뜻하지 않습니다.


In [ ]:
run_project("python", "scripts/check_retrieval.py", "--report", "data/processed/retrieval_demo/search-check.json")


## 4. 질문 입력

예: 한빛시청 도서관 통합 시스템의 사업 예산과 기간은?
이 셀을 다시 실행하면 다른 질문을 입력할 수 있습니다.
Streamlit 화면은 PC에서 실행하고, Colab에서는 아래 입력창으로 같은 검색 함수를 확인합니다.


In [ ]:
from IPython.display import Markdown, display

question = input("질문: ").strip()
if not question:
    raise ValueError("질문을 입력해 주세요.")
response = run_project("python", "scripts/retrieval_sample.py", "--config", "configs/retrieval_demo.yaml", "search", "--query", question, capture=True)
for rank, document in enumerate(json.loads(response.stdout), 1):
    display(Markdown(f"### {rank}. {document['metadata']['doc_id']}"))
    print("청크 ID:", document["metadata"]["chunk_id"])
    display(Markdown(document["page_content"]))
